In [ ]:
#importing libraries

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

#=============================================================================================================

#loading dataset

try:
    df = pd.read_excel('Fitness_Health_Tracking_Dataset_with_Missing_Values.xlsx')
    df_cleaned = df.dropna()
    print(f"Dataset loaded: {len(df)} rows, {len(df_cleaned)} after dropping NaN.")
except FileNotFoundError:
    raise FileNotFoundError("Excel file not found. Make sure 'Fitness_Health_Tracking_Dataset_with_Missing_Values.xlsx' is in the working directory.")
except Exception as e:
    raise RuntimeError(f"Failed to load dataset: {e}")

#=============================================================================================================

#Question: How does activity hours affect mood?

#average activity minutes based on mood bar graph

def getAverage(mood):
    total = 0
    i = 0
    try:
        array = df_cleaned[['Mood', 'Active_Minutes']].to_numpy()
    except KeyError as e:
        raise KeyError(f"Expected column missing from dataset: {e}")
    for value in array:
        if value[0] == mood:
            try:
                minutes = int(value[1])
            except (ValueError, TypeError):
                continue  # skip rows where Active_Minutes can't be converted
            total += minutes
            i += 1
    if i == 0:
        return 0
    return total / i

#getting all averages
try:
    sad_average = getAverage('Sad')
    nuetral_average = getAverage('Neutral')
    happy_average = getAverage('Happy')
    stressed_average = getAverage('Stressed')

    df_average_mood = pd.DataFrame({
        'Mood': ['Sad', 'Nuetral', 'Happy', 'Stressed'],
        'Average': [sad_average, nuetral_average, happy_average, stressed_average]
    })

    sns.barplot(data=df_average_mood, x='Mood', y='Average', hue='Mood', palette='pastel')
    plt.title("Average Active Minutes for Mood")
    plt.show()
except Exception as e:
    print(f"Error in mood vs active minutes chart: {e}")

#=============================================================================================================

#Question: How does workout type affect calories burned?

try:
    sns.scatterplot(df_cleaned.head(100), x='Calories_Burned', y='Active_Minutes', hue='Workout_Type')
    plt.show()
except KeyError as e:
    print(f"Missing column for scatter plot: {e}")
except Exception as e:
    print(f"Error generating scatter plot: {e}")

#=============================================================================================================

#Question: How does workout type affect heartrate?

def getAverageHeartRate(workout):
    total = 0
    i = 0
    try:
        array = df_cleaned[['Workout_Type', 'Heart_Rate (bpm)']].to_numpy()
    except KeyError as e:
        raise KeyError(f"Expected column missing from dataset: {e}")
    for value in array:
        if value[0] == workout:
            try:
                heart_rate = int(value[1])
            except (ValueError, TypeError):
                continue  # skip uncastable values
            total += heart_rate
            i += 1
    if i == 0:
        return 0
    return total / i

try:
    yoga_average = getAverageHeartRate('Yoga')
    cardio_average = getAverageHeartRate('Cardio')
    strength_average = getAverageHeartRate('Strength')

    df_average_workout = pd.DataFrame({
        'Workout_Type': ['Yoga', 'Cardio', 'Strength'],
        'Average': [yoga_average, cardio_average, strength_average]
    })

    sns.barplot(data=df_average_workout, x='Workout_Type', y='Average', hue='Workout_Type', palette='magma')
    plt.title("Average Heart Rate for Workout Type")
    plt.show()
except Exception as e:
    print(f"Error in workout type vs heart rate chart: {e}")

#=============================================================================================================

try:
    plt.hist(data=df_cleaned, x='Active_Minutes', edgecolor='black', bins=15)
    plt.title('Active Minutes')
    plt.show()
except Exception as e:
    print(f"Error plotting Active Minutes histogram: {e}")

#=============================================================================================================

try:
    plt.hist(data=df_cleaned, x='Steps_Taken', edgecolor='black', bins=15)
    plt.title('Steps Taken')
    plt.show()
except Exception as e:
    print(f"Error plotting Steps Taken histogram: {e}")

#=============================================================================================================

try:
    numeric_df = df.select_dtypes(include='number')
    corr = numeric_df.corr()
    corr
except Exception as e:
    print(f"Error computing correlation matrix: {e}")


In [ ]:
#=============================================================================================================

# CORRELATION ANALYSIS

from scipy import stats

try:
    numeric_df = df_cleaned.select_dtypes(include='number').drop(columns=['User_ID'])
except KeyError:
    numeric_df = df_cleaned.select_dtypes(include='number')
    print("Warning: 'User_ID' column not found, skipping drop.")
except Exception as e:
    raise RuntimeError(f"Failed to prepare numeric dataframe: {e}")

#=============================================================================================================

# Heatmap of all numeric correlations

try:
    plt.figure(figsize=(10, 8))
    corr_matrix = numeric_df.corr()
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(
        corr_matrix,
        mask=mask,
        annot=True,
        fmt='.2f',
        cmap='coolwarm',
        center=0,
        linewidths=0.5,
        square=True
    )
    plt.title('Correlation Heatmap of Numeric Variables', fontsize=14)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Error generating correlation heatmap: {e}")

#=============================================================================================================

# Top pairwise Pearson correlations

try:
    corr_unstacked = corr_matrix.unstack()
    corr_pairs = (
        corr_unstacked[corr_unstacked < 1.0]
        .reset_index()
    )
    corr_pairs.columns = ['Variable_1', 'Variable_2', 'Correlation']
    corr_pairs = corr_pairs[corr_pairs['Variable_1'] < corr_pairs['Variable_2']].copy()
    corr_pairs['Abs_Correlation'] = corr_pairs['Correlation'].abs()
    corr_pairs_sorted = corr_pairs.sort_values('Abs_Correlation', ascending=False).reset_index(drop=True)

    print("Top 10 Pairwise Correlations (by absolute value):")
    print(corr_pairs_sorted[['Variable_1', 'Variable_2', 'Correlation']].head(10).to_string(index=False))
except Exception as e:
    print(f"Error computing pairwise correlations: {e}")

#=============================================================================================================

# Bar chart of top correlations

try:
    top_n = corr_pairs_sorted.head(10).copy()
    top_n['Pair'] = top_n['Variable_1'] + ' vs\n' + top_n['Variable_2']
    colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in top_n['Correlation']]

    plt.figure(figsize=(12, 5))
    plt.barh(top_n['Pair'][::-1], top_n['Correlation'][::-1], color=colors[::-1], edgecolor='black')
    plt.axvline(0, color='black', linewidth=0.8)
    plt.xlabel('Pearson Correlation Coefficient')
    plt.title('Top 10 Pairwise Correlations')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Error generating top correlations bar chart: {e}")

#=============================================================================================================

# ANOVA: Does Workout Type significantly affect any numeric variable?

try:
    print("\nOne-Way ANOVA — Workout Type vs Numeric Variables")
    print(f"{'Variable':<30} {'F-statistic':>12} {'p-value':>10} {'Significant?':>14}")
    print("-" * 70)
    for col in numeric_df.columns:
        try:
            groups = [grp[col].values for _, grp in df_cleaned.groupby('Workout_Type')]
            f, p = stats.f_oneway(*groups)
            sig = 'Yes' if p < 0.05 else 'No'
            print(f"{col:<30} {f:>12.4f} {p:>10.4f} {sig:>14}")
        except Exception as col_err:
            print(f"{col:<30} {'ERROR':>12} — {col_err}")
except KeyError:
    print("Error: 'Workout_Type' column not found in dataset.")
except Exception as e:
    print(f"Error in Workout Type ANOVA: {e}")

#=============================================================================================================

# ANOVA: Does Mood significantly affect any numeric variable?

try:
    print("\nOne-Way ANOVA — Mood vs Numeric Variables")
    print(f"{'Variable':<30} {'F-statistic':>12} {'p-value':>10} {'Significant?':>14}")
    print("-" * 70)
    for col in numeric_df.columns:
        try:
            groups = [grp[col].values for _, grp in df_cleaned.groupby('Mood')]
            f, p = stats.f_oneway(*groups)
            sig = 'Yes' if p < 0.05 else 'No'
            print(f"{col:<30} {f:>12.4f} {p:>10.4f} {sig:>14}")
        except Exception as col_err:
            print(f"{col:<30} {'ERROR':>12} — {col_err}")
except KeyError:
    print("Error: 'Mood' column not found in dataset.")
except Exception as e:
    print(f"Error in Mood ANOVA: {e}")

#=============================================================================================================

# SUMMARY

try:
    print("\n=== CORRELATION SUMMARY ===")
    print(f"Total numeric variable pairs analysed: {len(corr_pairs_sorted)}")
    strong   = corr_pairs_sorted[corr_pairs_sorted['Abs_Correlation'] >= 0.3]
    moderate = corr_pairs_sorted[(corr_pairs_sorted['Abs_Correlation'] >= 0.1) & (corr_pairs_sorted['Abs_Correlation'] < 0.3)]
    weak     = corr_pairs_sorted[corr_pairs_sorted['Abs_Correlation'] < 0.1]
    print(f"  Strong correlations  (|r| >= 0.3): {len(strong)}")
    print(f"  Moderate correlations (0.1 <= |r| < 0.3): {len(moderate)}")
    print(f"  Weak correlations    (|r| < 0.1): {len(weak)}")
    print()
    print("Finding: Nearly all correlations are near zero. This confirms the dataset")
    print("is synthetically generated with no meaningful real-world relationships")
    print("embedded between variables.")
except Exception as e:
    print(f"Error generating correlation summary: {e}")


In [ ]:
#=============================================================================================================

# LINEAR REGRESSION — Predicting Calories Burned

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

#=============================================================================================================

def buildCalorieModel(dataframe):
    try:
        features = dataframe[['Active_Minutes', 'Steps_Taken']].to_numpy()
        target = dataframe['Calories_Burned'].to_numpy()
    except KeyError as e:
        raise KeyError(f"Missing required column for regression model: {e}")

    if len(features) < 5:
        raise ValueError("Not enough data to build a regression model (need at least 5 rows).")

    try:
        X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

        model = LinearRegression()
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        rmse = mean_squared_error(y_test, y_pred) ** 0.5
        r2 = r2_score(y_test, y_pred)

        print(f"Linear Regression — Predicting Calories Burned")
        print(f"  Coefficients: Active_Minutes={model.coef_[0]:.4f}, Steps_Taken={model.coef_[1]:.4f}")
        print(f"  Intercept: {model.intercept_:.4f}")
        print(f"  RMSE: {rmse:.4f}")
        print(f"  R² Score: {r2:.4f}")
    except Exception as e:
        print(f"Error training regression model: {e}")
        return

    try:
        plt.figure(figsize=(7, 5))
        plt.scatter(y_test, y_pred, alpha=0.5, edgecolors='black', linewidths=0.4)
        plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=1.5)
        plt.xlabel('Actual Calories Burned')
        plt.ylabel('Predicted Calories Burned')
        plt.title('Actual vs Predicted Calories Burned (Linear Regression)')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Error plotting regression results: {e}")

try:
    buildCalorieModel(df_cleaned)
except (KeyError, ValueError) as e:
    print(f"Could not build calorie model: {e}")
except Exception as e:
    print(f"Unexpected error in buildCalorieModel: {e}")

#=============================================================================================================
